<font size="6" color="blue"><b>Module 5 – Support Vector Machines</b></font>

<img src="images_module05/module05.png" width="600"/>


# Module Overview
***

Support Vector Machines (SVMs) are powerful and versatile Machine Learning models, capable of performing linear or nonlinear classification, regression, and even outlier detection. SVMs are particularly well suited for classification of complex **small-or medium-sized datasets.** 

**Learning Goal:** To understand how to apply Support Vectors Machines to solve regression and classification problems; this includes the underlying theory (margin maximization, quadratic programming, etc.), as well as how to implement them using Sklearns SVM module. 

**Additional Resources:**

Quadatric Programming & Optimization Links:
- https://cvxopt.org/
- https://cvxopt.org/userguide/coneprog.html#quadratic-programming
- https://cvxopt.org/examples/tutorial/qp.html
- https://courses.csail.mit.edu/6.867/wiki/images/a/a7/Qp-cvxopt.pdf
- https://www.youtube.com/watch?v=GZb9647X8sg
- https://www.youtube.com/watch?v=oaiiyIsbNdI
- https://en.wikipedia.org/wiki/Quadratic_programming
- https://www.svm-tutorial.com/2016/09/duality-lagrange-multipliers/
- https://scaron.info/blog/quadratic-programming-in-python.html

SVM Theory:
- https://www.mathworks.com/help/stats/understanding-support-vector-machine-regression.html
- http://web.mit.edu/6.034/wwwbob/svm-notes-long-08.pdf
- https://towardsdatascience.com/understanding-support-vector-machine-part-1-lagrange-multipliers-5c24a52ffc5e
- https://medium.com/deep-math-machine-learning-ai/chapter-3-support-vector-machine-with-math-47d6193c82be
- http://web.mit.edu/6.034/wwwbob/svm-notes-long-08.pdf
- https://www.quora.com/What-is-primal-and-dual-formulation-in-SVM
- https://pythonprogramming.net/why-use-kernel-with-svm-machine-learning-tutorial/?completed=/kernels-with-svm-machine-learning-tutorial/
- http://cs229.stanford.edu/notes/cs229-notes3.pdf

SVM Custom Implementations:
- https://xavierbourretsicotte.github.io/SVM_implementation.html 
- https://pythonprogramming.net/soft-margin-kernel-cvxopt-svm-machine-learning-tutorial/



In [1]:
# Common imports
import numpy as np
import os
import tarfile
from six.moves import urllib
import pandas as pd
import sklearn

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

ModuleNotFoundError: No module named 'sklearn'

# Linear SVMs

## Classification

SVM Classifiers work by fitting the **widest possible margin between any two classes**. 
- Whereas a logistic regression would simply draw a single line between two classes, an SVM draws **two parallel lines and attempts to push them as far apart as possible.** 

This is pretty vague – SVMs are best described via pictures. Let's generate some predictions using SVMs on the Iris flower dataset we used in the last module:

In [ ]:
# Import data
from sklearn.svm import SVC
from sklearn import datasets

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = iris["target"]

print(X.shape)
print(y.shape)
print(np.unique(y))

# Plot setosa and versicolor
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo");

In [ ]:
# Train a model
setosa_or_versicolor = (y == 0) | (y == 1)
X = X[setosa_or_versicolor]
y = y[setosa_or_versicolor]

# SVM Classifier model
svm_clf = SVC(kernel="linear", C=100000000)
svm_clf.fit(X, y)

In [ ]:
# Bad models
x0 = np.linspace(0, 5.5, 200)
pred_1 = 5*x0 - 20
pred_2 = x0 - 1.8
pred_3 = 0.1 * x0 + 0.5

def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    w = svm_clf.coef_[0]
    b = svm_clf.intercept_[0]

    # At the decision boundary, w0*x0 + w1*x1 + b = 0
    # => x1 = -w0/w1 * x0 - b/w1 
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0]/w[1] * x0 - b/w[1]

    margin = 1/w[1] # Margin is constant (linear)
    gutter_up = decision_boundary + margin
    gutter_down = decision_boundary - margin

    svs = svm_clf.support_vectors_
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#FFAAAA')
    plt.plot(x0, decision_boundary, "k-", linewidth=2)
    plt.plot(x0, gutter_up, "k--", linewidth=2)
    plt.plot(x0, gutter_down, "k--", linewidth=2)

fig, axes = plt.subplots(ncols=2, figsize=(10,2.7), sharey=True)

plt.sca(axes[0])
plt.plot(x0, pred_1, "g--", linewidth=2)
plt.plot(x0, pred_2, "m-", linewidth=2)
plt.plot(x0, pred_3, "r-", linewidth=2)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", label="Iris setosa")
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.legend(loc="upper left", fontsize=14)
plt.axis([0, 5.5, 0, 2])
plt.title('Bad models')

plt.sca(axes[1])
plot_svc_decision_boundary(svm_clf, 0, 5.5)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo")
plt.xlabel("Petal length", fontsize=14)
plt.axis([0, 5.5, 0, 2])
plt.title('SVM model')
plt.show()

The two classes can clearly be **separated easily with a straight line (they are linearly separable).** 

The left plot (non-SVM):
- The decision boundaries of three possible linear classifiers. 
- Green dotted plot is pretty terrible
- The other two models work perfectly on this training set.
- HOWEVER, **their decision boundaries come so close to the instances that these models will probably not perform as well on new instances.** 

The right plot (SVM):
- The solid line represents the decision boundary of an SVM classifier
- this line not only separates the two classes but also **stays as far away from the closest training instances as possible.** 
- You can think of an SVM classifier as fitting the widest possible street (represented by the parallel dashed lines) between the classes. **This is called large margin classification.**


**NOTE:** Adding more training instances **“off the street”** will not affect the decision boundary at all: 
- it is fully determined (or “supported”) by the instances located on the edge of the street. 
- These instances are called the **support vectors**.


In [ ]:
w = svm_clf.coef_[0]
b = svm_clf.intercept_[0]
svs = svm_clf.support_vectors_

print("COEFs:\n", w,b)
print("SVs:\n", svs)
print("Decision Function at SVs:")
for sv in svm_clf.support_vectors_:
    print(svm_clf.decision_function(sv.reshape(-1,2)))

SVMs are sensitive to the feature scales, as you can see in the in the left plot below.

The vertical scale is much larger than the horizontal scale, so the widest possible street is close to horizontal.


In [ ]:
from sklearn.svm import SVC

Xs = np.array([[1, 50], [5, 20], [3, 80], [5, 60]]).astype(np.float64)
ys = np.array([0, 0, 1, 1])
svm_clf = SVC(kernel="linear", C=100)
svm_clf.fit(Xs, ys)

plt.figure(figsize=(9,2.7))
plt.subplot(121)
plt.plot(Xs[:, 0][ys==1], Xs[:, 1][ys==1], "bo")
plt.plot(Xs[:, 0][ys==0], Xs[:, 1][ys==0], "ms")
plot_svc_decision_boundary(svm_clf, 0, 6)
plt.xlabel("$x_0$", fontsize=20)
plt.ylabel("$x_1$    ", fontsize=20, rotation=0)
plt.title("Unscaled", fontsize=16)
plt.axis([0, 6, 0, 90])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(Xs)
svm_clf.fit(X_scaled, ys)

plt.subplot(122)
plt.plot(X_scaled[:, 0][ys==1], X_scaled[:, 1][ys==1], "bo")
plt.plot(X_scaled[:, 0][ys==0], X_scaled[:, 1][ys==0], "ms")
plot_svc_decision_boundary(svm_clf, -2, 2)
plt.xlabel("$x_0$", fontsize=20)
plt.ylabel("$x'_1$  ", fontsize=20, rotation=0)
plt.title("Scaled", fontsize=16)
plt.axis([-2, 2, -2, 2])

**Soft Margin Classification**

The method we used above is called **Hard margin classification:** 
- We strictly impose that all instances must be off the street and on the correct side. 
- This method only works if the data is **linearly separable**. 
- It is also very **sensitive to outliers.**

If we add an outlier over the decision boundary, the margins become squeezed, and the model will not generalize as well:

In [ ]:
from sklearn.svm import SVC

X_outlier = np.array([[3.2, 0.8]]) # Add impossible outlier
y_outlier = np.array([0])
Xo1 = np.concatenate([X, X_outlier], axis=0)
yo1 = np.concatenate([y, y_outlier], axis=0)

svm_clf = SVC(kernel="linear", C=10**9)
svm_clf.fit(Xo1, yo1)

plt.plot(Xo1[:, 0][yo1==1], Xo1[:, 1][yo1==1], "bs")
plt.plot(Xo1[:, 0][yo1==0], Xo1[:, 1][yo1==0], "yo")
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.annotate("Outlier", xy=(X_outlier[0][0], X_outlier[0][1]), 
             xytext=(2.5, 1.7), 
             ha="center", 
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=16)
plot_svc_decision_boundary(svm_clf, 0, 5.5)

plt.axis([0, 5.5, 0, 2])

In [ ]:
svm_clf.support_vectors_

In [ ]:
for sv in svm_clf.support_vectors_:
    print(svm_clf.decision_function(sv.reshape(-1,2)))

If we move the outlier even further to the right so that the classes are **not linearly seperable**, it become impossible to fit a hard margin:

In [ ]:
from sklearn.svm import SVC

X_outlier = np.array([[3.4, 1.3]]) # Add impossible outlier
y_outlier = np.array([0])
Xo1 = np.concatenate([X, X_outlier], axis=0)
yo1 = np.concatenate([y, y_outlier], axis=0)

svm_clf = SVC(kernel="linear", C=10**9) # High C approaches hard margin
svm_clf.fit(Xo1, yo1)

plt.plot(Xo1[:, 0][yo1==1], Xo1[:, 1][yo1==1], "bs")
plt.plot(Xo1[:, 0][yo1==0], Xo1[:, 1][yo1==0], "yo")
plt.text(0.3, 1.0, "Impossible!", fontsize=24, color="red")
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.annotate("Outlier", xy=(X_outlier[0][0], X_outlier[0][1]), 
             xytext=(2.5, 1.7), 
             ha="center", 
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=16)
plot_svc_decision_boundary(svm_clf, 0, 5.5)

plt.axis([0, 5.5, 0, 2])

To avoid these issues, use **a more flexible model.** The objective is to find a good balance between:
- keeping the street as large as possible, and
- limiting the margin violations (i.e., instances that end up in the middle of the street or even on the wrong side). 

This is called **soft margin classification.** 

When creating an SVM model with `sklearn`, we can control how "soft" the margin are with the `C` hyperparameter: 
- **Low C value:** many margin violations will be allowed and the model will be LESS CONSTRAINED **(softer margin, more regularized)**
- **High C value:** fewer margin violations will be allowed  and the model will be MORE CONSTRAINED **(harder margin, less regularized)**

Generally speaking, **margin violations are bad**; however, when your data has many outliers, allowing margin violations can allow the model to generalize better. Below we see some examples of this.

**NOTE:** Below we use the `LinearSVC` class, which is practically equivalent to using the `SVC` class with a `kernel="linear"`. 
- When creating the SVC model, we would write `SVC(kernel="linear", C=1)` . 
- Or we could use the `SGDClassifier` class, with `SGDClassifier(loss="hinge", alpha=1/(m*C))` . This applies regular Stochastic Gradient Descent to train a linear SVM classifier.

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = (iris["target"] == 2).astype(np.float64)  # Iris virginica

scaler = StandardScaler()

# Make two models with different C values, small and large
svm_clf1 = LinearSVC(C=1, loss="hinge", random_state=42)
svm_clf2 = LinearSVC(C=110, loss="hinge", random_state=42)

scaled_svm_clf1 = Pipeline([
        ("scaler", scaler),
        ("linear_svc", svm_clf1),
    ])
scaled_svm_clf2 = Pipeline([
        ("scaler", scaler),
        ("linear_svc", svm_clf2),
    ])

scaled_svm_clf1.fit(X, y)
scaled_svm_clf2.fit(X, y)

In [ ]:
# Convert to unscaled parameters
b1 = svm_clf1.decision_function([-scaler.mean_ / scaler.scale_])
b2 = svm_clf2.decision_function([-scaler.mean_ / scaler.scale_])
w1 = svm_clf1.coef_[0] / scaler.scale_
w2 = svm_clf2.coef_[0] / scaler.scale_
svm_clf1.intercept_ = np.array([b1])
svm_clf2.intercept_ = np.array([b2])
svm_clf1.coef_ = np.array([w1])
svm_clf2.coef_ = np.array([w2])

# Find support vectors (LinearSVC does not do this automatically)
t = y * 2 - 1
support_vectors_idx1 = (t * (X.dot(w1) + b1) < 1).ravel()
support_vectors_idx2 = (t * (X.dot(w2) + b2) < 1).ravel()
svm_clf1.support_vectors_ = X[support_vectors_idx1]
svm_clf2.support_vectors_ = X[support_vectors_idx2]

fig, axes = plt.subplots(ncols=2, figsize=(10,2.7), sharey=True)

plt.sca(axes[0])
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^", label="Iris virginica")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs", label="Iris versicolor")
plot_svc_decision_boundary(svm_clf1, 4, 5.9)
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.legend(loc="upper left", fontsize=14)
plt.title("$C = {}$".format(svm_clf1.C), fontsize=16)
plt.axis([4, 5.9, 0.8, 2.8])

plt.sca(axes[1])
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
plot_svc_decision_boundary(svm_clf2, 4, 5.99)
plt.xlabel("Petal length", fontsize=14)
plt.title("$C = {}$".format(svm_clf2.C), fontsize=16)
plt.axis([4, 5.9, 0.8, 2.8])

In [ ]:
svm_clf2.support_vectors_

In [ ]:
from sklearn.svm import SVC
from sklearn import datasets

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)] # petal length, petal width
y = (iris["target"] == 2).astype(np.float64) # Iris virginica

svm_clf = SVC(kernel="linear", C=1)
svm_clf.fit(X, y)
svm_clf.predict([[5.3, 1.3]])

## Regression

To use SVMs for regression instead of classification (for linear data), the trick is to reverse the objective: 
- instead of trying to fit the largest possible street between two classes while limiting margin violations, SVM Regression tries to **fit as many instances as possible on the street while limiting margin violations (i.e., instances off the street).**

You can use Scikit-Learn’s `LinearSVR` class to perform linear SVM Regression. 
- The width of the street is controlled by a hyperparameter, ϵ. 
- The plots below show two linear SVM Regression models trained on some random linear data, one with a large margin ($ϵ = 2.5$) and the other with a small margin ($ϵ = 0.5$). 


In [ ]:
np.random.seed(42)
m = 50
X = 2 * np.random.rand(m, 1)
y = (4 + 3 * X + np.random.randn(m, 1)).ravel()

plt.scatter(X,y);

In [ ]:
from sklearn.svm import LinearSVR

svm_reg = LinearSVR(epsilon=1.5, random_state=42)
svm_reg.fit(X, y)

In [ ]:
svm_reg1 = LinearSVR(epsilon=2.5, random_state=42, intercept_scaling=10)
svm_reg2 = LinearSVR(epsilon=0.5, random_state=42)
svm_reg1.fit(X, y)
svm_reg2.fit(X, y)

def find_support_vectors(svm_reg, X, y):
    y_pred = svm_reg.predict(X)
    off_margin = (np.abs(y - y_pred) >= svm_reg.epsilon)
    return np.argwhere(off_margin)

svm_reg1.support_ = find_support_vectors(svm_reg1, X, y)
svm_reg2.support_ = find_support_vectors(svm_reg2, X, y)

eps_x1 = 1
eps_y_pred = svm_reg1.predict([[eps_x1]])

In [ ]:
def plot_svm_regression(svm_reg, X, y, axes):
    m = X.shape[0]
    x1s = np.linspace(axes[0], axes[1], m).reshape(m, 1)
    y_pred = svm_reg.predict(x1s)
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$")
    plt.plot(x1s, y_pred + svm_reg.epsilon, "k--")
    plt.plot(x1s, y_pred - svm_reg.epsilon, "k--")
    plt.scatter(X[svm_reg.support_], y[svm_reg.support_], s=180, facecolors='#FFAAAA')
    plt.plot(X, y, "bo")
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.legend(loc="upper left", fontsize=18)
    plt.axis(axes)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_reg1, X, y, [0, 2, 3, 12])
plt.title(r"$\epsilon = {}$".format(svm_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
#plt.plot([eps_x1, eps_x1], [eps_y_pred, eps_y_pred - svm_reg1.epsilon], "k-", linewidth=2)
plt.annotate(
        '', xy=(eps_x1, eps_y_pred), xycoords='data',
        xytext=(eps_x1, eps_y_pred - svm_reg1.epsilon),
        textcoords='data', arrowprops={'arrowstyle': '<->', 'linewidth': 1.5}
    )
plt.text(0.91, 5.6, r"$\epsilon$", fontsize=20)
plt.sca(axes[1])
plot_svm_regression(svm_reg2, X, y, [0, 2, 3, 12])
plt.title(r"$\epsilon = {}$".format(svm_reg2.epsilon), fontsize=18)
plt.show()

Adding more training instances within the margin does not affect the model’s predictions; thus, the model is said to be **ϵ-insensitive.** 


# Training Linear SVMs

## Decision function

The linear SVM classifier model predicts the class of a new instance $x$ by simply computing the decision function: 

$w^Tx + b = w_1x_1 + ... + w_nx_n + b $

- The decision boundary, that separates both classes is the set of points where:
    - $w^T \cdot x + b=0$
    
    
- The margins (dashed lines) are the set of points where:
    - $w^T \cdot x + b=1$
    - $w^T \cdot x + b=-1$


The SVM algorithm finds $w$ and $b$, so that the margin is as wide as possible while avoiding margin violations (hard margin) or limiting them (soft margin).

For training purposes, we **assign all positive instances to 1, and all negative instances to -1:**

$y^{(i)}=\left\{
                \begin{array}{ll}
                  -1 &\text{ if } \mathbf{w^T}\mathbf{x}^{(i)}+b \leq -1 \\
                  1 &\text{ if } \mathbf{w^T}\mathbf{x}^{(i)}+b \ge 1 \\
                \end{array}
              \right.$
              
which can be more concisely written as:

$y^{(i)} (\mathbf{w^T}\mathbf{x}^{(i)}+b) \ge 1$

Let:

- $\mathbf{𝐱}$ be a feature vector (i.e., the input of the SVM). $\mathbf{x} \in \mathbb{R}^n$, where 𝑛 is the dimension of the feature vector.
- $\mathbf{y}$ be the class (i.e., the output of the SVM). $y \in \{ -1,1\}$, i.e. the classification task is binary.
- $\mathbf{w}$ and $b$ be the parameters of the SVM: we need to learn them using the training set.
- $(\mathbf{x}^{(i)}, y^{(i)})$ be the 𝑖th sample in the dataset. Let's assume we have 𝑁 samples in the training set.

With 𝑛=2, one can represent the SVM's decision boundaries as follows:

<img src="images_module05/01_svbdf.png" width="400"/>

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = (iris["target"] == 2).astype(np.float64)  # Iris virginica

def plot_3D_decision_function(ax, w, b, x1_lim=[4, 6], x2_lim=[0.8, 2.8]):
    x1_in_bounds = (X[:, 0] > x1_lim[0]) & (X[:, 0] < x1_lim[1])
    X_crop = X[x1_in_bounds]
    y_crop = y[x1_in_bounds]
    x1s = np.linspace(x1_lim[0], x1_lim[1], 20)
    x2s = np.linspace(x2_lim[0], x2_lim[1], 20)
    x1, x2 = np.meshgrid(x1s, x2s)
    xs = np.c_[x1.ravel(), x2.ravel()]
    df = (xs.dot(w) + b).reshape(x1.shape)
    m = 1 / np.linalg.norm(w)
    boundary_x2s = -x1s*(w[0]/w[1])-b/w[1]
    margin_x2s_1 = -x1s*(w[0]/w[1])-(b-1)/w[1]
    margin_x2s_2 = -x1s*(w[0]/w[1])-(b+1)/w[1]
    ax.plot_surface(x1s, x2, np.zeros_like(x1),
                    color="b", alpha=0.2, cstride=100, rstride=100)
    ax.plot(x1s, boundary_x2s, 0, "k-", linewidth=2, label=r"$h=0$")
    ax.plot(x1s, margin_x2s_1, 0, "k--", linewidth=2, label=r"$h=\pm 1$")
    ax.plot(x1s, margin_x2s_2, 0, "k--", linewidth=2)
    ax.plot(X_crop[:, 0][y_crop==1], X_crop[:, 1][y_crop==1], 0, "g^")
    ax.plot_wireframe(x1, x2, df, alpha=0.3, color="k")
    ax.plot(X_crop[:, 0][y_crop==0], X_crop[:, 1][y_crop==0], 0, "bs")
    ax.axis(x1_lim + x2_lim + [-7.5, 10])
    ax.text(4.5, 2.5, 3.8, "Decision function $h$", fontsize=16)
    ax.set_xlabel(r"Petal length", fontsize=16, labelpad=10)
    ax.set_ylabel(r"Petal width", fontsize=16, labelpad=10)
    ax.set_zlabel(r"$h = \mathbf{w}^T \mathbf{x} + b$", fontsize=18, labelpad=5)
    ax.legend(loc="upper left", fontsize=16)

fig = plt.figure(figsize=(11, 6))
ax1 = fig.add_subplot(111, projection='3d')
plot_3D_decision_function(ax1, w=svm_clf2.coef_[0], b=svm_clf2.intercept_[0])
plt.show()

The data rests on a 2D plane because this dataset has two features (petal width and petal length). 

The decision boundary is the set of points where the decision function is equal to 0: 
- **it is the intersection of two planes,** which is a straight line (represented by the thick solid line).  

The dashed lines represent the points where the decision function is equal to 1 or –1:
- they are parallel and at equal distance to the decision boundary and they form a margin around it. 
- Training a linear SVM classifier means finding the values of $w$ and $b$ that make this margin as wide as possible while avoiding margin violations (hard margin) or limiting them (soft margin). 



## Training Objective


The SVM aims at satisfying two requirements:

1. The SVM should maximize the distance between the two decision boundaries: 
    - This means we want to maximize the distance between the hyperplane defined by $\mathbf{w^T}\mathbf{x}+b = -1$ and the hyperplane defined by $\mathbf{w^T}\mathbf{x}+b = 1$. 
    - This distance is equal to $\frac{2}{\|\mathbf{w}\|}$ (derived from distance between point and plane, see https://web.mit.edu/6.034/wwwbob/svm-notes-long-08.pdf)
    - This means we want to solve $\underset{\mathbf{w}}{\operatorname{max}} \frac{2}{\|\mathbf{w}\|}$. Equivalently we want $\underset{\mathbf{w}}{\operatorname{min}} \frac{\|\mathbf{w}\|}{2}$.


2. The SVM should also correctly classify all 𝐱(𝑖):
    - which means $y^{(i)} (\mathbf{w^T}\mathbf{x}^{(i)}+b) \ge 1, \forall i \in \{1,\dots,N\}$

Which leads us to the following **quadratic optimization** problem. Note that this is the hard-margin SVM, as this quadratic optimization problem admits a solution **only if the data is linearly separable:**


$\begin{align}
\min_{\mathbf{w},b}\quad &\frac{\|\mathbf{w}\|}{2}, \\
s.t.\quad&y^{(i)} (\mathbf{w^T}\mathbf{x}^{(i)}+b) \ge 1 &\forall i \in \{1,\dots,N\}
\end{align}$



**NOTE:** We actually minimize $\frac{1}{2}w^Tw $, which is equal to $\frac{1}{2}\|w\|^2 $, rather than $\frac{\|\mathbf{w}\|}{2}$ because $\frac{1}{2}w^Tw $ has a nice and simple derivative (which is just $\mathbf{w}$). Also $\|w\|$ is not differentiable at 0. We can also wite this problem as follows:


$ \text{Hard-margin:} \begin{cases} 
      \text{minimize} & \frac{1}{2}w^T \cdot w \\
      \text{subject to} & y^{(i)}(w^T \cdot x^{(i)}+b)\geq 1 \text{ for every training example } 
   \end{cases}
$

Vector derivative rules: http://www.gatsby.ucl.ac.uk/teaching/courses/sntn/sntn-2017/resources/Matrix_derivatives_cribsheet.pdf

Another way to look at this problem is by considering the slope of the decision function $w^Tx + b$:

- It is equal to the norm of the weight vector, $\|\mathbf{w}\|$. 
- If we divide this slope by 2, the points where the decision function is equal to $±1$ are going to be twice as far away from the decision boundary. 

In other words, **dividing the slope by 2 will multiply the margin by 2**:
- the smaller the weight vector $\mathbf{w}$, the larger the margin. 



In [ ]:
def plot_2D_decision_function(w, b, ylabel=True, x1_lim=[-3, 3]):
    x1 = np.linspace(x1_lim[0], x1_lim[1], 200)
    y = w * x1 + b
    m = 1 / w

    plt.plot(x1, y)
    plt.plot(x1_lim, [1, 1], "k:")
    plt.plot(x1_lim, [-1, -1], "k:")
    plt.axhline(y=0, color='k')
    plt.axvline(x=0, color='k')
    plt.plot([m, m], [0, 1], "k--")
    plt.plot([-m, -m], [0, -1], "k--")
    plt.plot([-m, m], [0, 0], "k-o", linewidth=3)
    plt.axis(x1_lim + [-2, 2])
    plt.xlabel(r"$x_1$", fontsize=16)
    if ylabel:
        plt.ylabel(r"$w_1 x_1$  ", rotation=0, fontsize=16)
    plt.title(r"$w_1 = {}$".format(w), fontsize=16)

fig, axes = plt.subplots(ncols=2, figsize=(9, 3.2), sharey=True)
plt.sca(axes[0])
plot_2D_decision_function(1, 0)
plt.sca(axes[1])
plot_2D_decision_function(0.5, 0, ylabel=False)
plt.show()

## Slack variable for soft margin classification

We can relax the constraints by introducing so-called slack variables $\zeta^{( i )} ≥ 0$, which measures how much the $i^{th}$ instance is allowed to violate the margin. 

Each sample of the training set has its own slack variable; **the higher the slack variable, the more instances will be able to cross the margin thresholds.**

We now have two conflicting objectives: 
- make the slack variables as small as possible to reduce the margin violations
- make $w^⊺w$ as small as possible to increase the margin. 

This is where the $C$ hyperparameter comes in. $C$ allows us to define the tradeoff between these two objectives. This gives us the constrained optimization problem we see below:

- **Hard Margin Problem:**

$$ \text{Hard-margin:} \begin{cases} 
      \text{minimize} & \frac{1}{2}w^T \cdot w \\
      \text{subject to} & y^{(i)}(w^T \cdot x^{(i)}+b)\geq 1 \text{ for every training example } 
   \end{cases}
$$

- **Soft Margin Problem:**



$$ \text{Soft-margin:} \begin{cases} 
      \text{minimize} & \frac{1}{2}w^T \cdot w +C\sum_{i=1}^m \xi^{(i)} \\
      \text{subject to} & y^{(i)}(w^T \cdot x^{(i)}+b)\geq 1 - \xi^{(i)} \text{, for every $i$ training example } 
   \end{cases}
$$


**The higher $C$ is, the greater influence the slack parameter will have, and the harder our margins will be.**


## Quadratic Programming

The hard margin and soft margin problems are both **convex quadratic optimization problems with linear constraints.** Such problems are known as **Quadratic Programming (QP)** problems. 

Many off-the-shelf solvers are available to solve QP problems, but most are out of scope of this course. 




In its most general formulation, quadratic programming looks like:

<img src="images_module05/02_qp.png" width="600"/>

Note that the expression $\mathbf{Ap} ≤ b$ defines $n_c$ constraints: 
- $p^Ta^{(i)} ≤ b^{(i)}$ for $i = 1, 2,..., n_c$, 
- where $a^{( i )}$ is the vector containing the elements of the $i^{th}$ row of $A$ and $b^{(i)}$ is the $i^{th}$ element of $b$.

Doing the appropriate substitutions, the hard-margin formulation can be seen as a QP problem:

- $p = (b,w_1,...,w_n)$, our parameter vector

- $n_p=n+1$, the number of features + 1, to include the bias term $b$

- $n_c = m$, where m is the number of training instances.

- $H$ is the $n_p × n_p$ identity matrix, except with a zero in the top-left cell (to ignore the bias term).

- $f = 0$, an $n_p$ dimensional vector full of 0s.

- $b$ = –1, an $n_c$ dimensional vector full of –1s.

- $a^{( i )} = – y^{( i )} X^{( i )}$ , where $X^{( i )}$ is equal to $x^{( i )}$ with an extra bias feature $X_0 = 1$.

Once a solution $p$ for the QP problem is found, then the SVM solution is $b=p_0$ and $w=(u_1,...,u_n)$.

This formulation is called _primal_. There is an equivalent formulation, called _dual_, which allows SVM to be extended to using non-linear kernels (which we will discuss shortly).


## CVXOPT Implementation

In `cvxopt.solvers.qp` docs, notation looks like this:

    minimize    (1/2)*x'*P*x + q'*x
    subject to  G*x <= h
                A*x = b.


In [ ]:
!conda install cvxopt

In [ ]:
import cvxopt
?cvxopt.solvers.qp

To make this work with `CVXOPT`, we can reform the hard margin problem to fit he quadratic form:

**Hard margin form:**

$ \text{Hard-margin:} \begin{cases} 
      \text{minimize} & \frac{1}{2}w^T \cdot w \\
      \text{subject to} & y^{(i)}(w^T \cdot x^{(i)}+b)\geq 1 \text{ for every training example } 
   \end{cases}
$

**Basic Primal Problem:**

$\begin{aligned}
    & \min \frac{1}{2} x^TPx + q^Tx
    \\
     s.t. \ & \ Gx \leq h 
    \\
    & \ Ax = b
\end{aligned}$

**CVXOPT QP Solver function**:

- `cvxopt.solvers.qp(P, q[, G, h[, A, b[, solver[, initvals]]]])`
- Optimization methods: https://en.wikipedia.org/wiki/Quadratic_programming

In [ ]:
# Import data
from sklearn.svm import SVC
from sklearn import datasets

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = iris["target"]

setosa_or_versicolor = (y == 0) | (y == 1)
X = X[setosa_or_versicolor]
y = y[setosa_or_versicolor]

print("X shape:", X.shape)
print("y shape:",y.shape)
print("y unique:",np.unique(y))

plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo")
plt.show()

# Fit a model
svm_clf = SVC(kernel="linear", C=100)
svm_clf.fit(X, y)

w = svm_clf.coef_[0]
b = svm_clf.intercept_[0]
svs = svm_clf.support_vectors_

print("COEFs:\n", w,b)
print("SVs:\n", svs)

for sv in svm_clf.support_vectors_:
    print(svm_clf.decision_function(sv.reshape(-1,2)))

In [ ]:
#Importing with custom names to avoid issues with numpy / sympy matrix
from cvxopt import matrix as cvxopt_matrix
from cvxopt import solvers as cvxopt_solvers

#Initializing values and computing 
m, n = X.shape
X_b = np.c_[np.ones(m), X]

# P = Identity with 0 at [0,0] for bias
P = np.identity(n+1)
P[0,0] = 0
print('P:', P)

# q = 0
q = np.zeros(n+1)
print('q:', q[0:5])

# G = X*-y
y = y.reshape(-1,1) * 1.
y[y==0] = -1
G = X_b*-y
print('G:', G.shape)

# h = -1
h = -np.ones(m)
print('h:', h[0:5])

In [ ]:
#Converting into cvxopt format
P = cvxopt_matrix(P)
q = cvxopt_matrix(q)
G = cvxopt_matrix(G)
h = cvxopt_matrix(h)

#Setting solver parameters (change default to decrease tolerance) 
cvxopt_solvers.options['show_progress'] = False
cvxopt_solvers.options['abstol'] = 1e-10
cvxopt_solvers.options['reltol'] = 1e-10
cvxopt_solvers.options['feastol'] = 1e-10

#Run solver
sol = cvxopt_solvers.qp(P, q, G, h)
qp_w = np.array(sol['x'])
qp_w

# Extract indexs of support vectors
for idx, i in enumerate(np.array(sol['z'])>0.001):
    if i == True:
        print("SUPPORT VECTOR:", idx)

sol

In [ ]:
# Compare results

print("COEFs svm:\n", w,b)
print("COEFs qps:\n", qp_w[1:].reshape(-1,), qp_w[0])

print("SVs svm:\n", svs)
print("SVs qps:\n", X[[44, 98],:])

# Non-linear SVMs

## Polynomial Features

As we know, most data is not completely linearly separable. 

One approach to handling nonlinear datasets is to add more features, such as **polynomial features**; in some cases this can result in a **linearly separable dataset.**

As you see in the plot below, adding a 2nd degree polynomial feature to the 1 dimensional dataset creates a linearly separable dataset:

In [ ]:
X1D = np.linspace(-4, 4, 9).reshape(-1, 1)
X2D = np.c_[X1D, X1D**2]
y = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

plt.figure(figsize=(10, 3))

plt.subplot(121)
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.plot(X1D[:, 0][y==0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][y==1], np.zeros(5), "g^")
plt.gca().get_yaxis().set_ticks([])
plt.xlabel(r"$x_1$", fontsize=20)
plt.axis([-4.5, 4.5, -0.2, 0.2])

plt.subplot(122)
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(X2D[:, 0][y==0], X2D[:, 1][y==0], "bs")
plt.plot(X2D[:, 0][y==1], X2D[:, 1][y==1], "g^")
plt.xlabel(r"$x_1$", fontsize=20)
plt.ylabel(r"$x_2$  ", fontsize=20, rotation=0)
plt.gca().get_yaxis().set_ticks([0, 4, 8, 12, 16])
plt.plot([-4.5, 4.5], [6.5, 6.5], "r--", linewidth=3)
plt.axis([-4.5, 4.5, -1, 17])

plt.subplots_adjust(right=1)
plt.show()

Let’s test out this idea futher using the moons dataset: 
- this is a toy dataset for binary classification in which the data points are shaped as two interleaving half circles:

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=200, noise=0.2, random_state=42)

def plot_dataset(X, y, axes):
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
    plt.axis(axes)
    plt.grid(True, which='both')
    plt.xlabel(r"$x_1$", fontsize=20)
    plt.ylabel(r"$x_2$", fontsize=20, rotation=0)

plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.show()

## Polynomial Kernel

Adding polynomial features is simple to implement and can work great with all sorts of Machine Learning algorithms (not just SVMs).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=5)
pf.fit_transform(X).shape

In [ ]:
from sklearn.datasets import make_moons
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC

polynomial_svm_clf = Pipeline([
        ("poly_features", PolynomialFeatures(degree=10)),
        ("scaler", StandardScaler()),
        ("svm_clf", LinearSVC(C=0.1, loss="hinge", random_state=42))
    ])

polynomial_svm_clf.fit(X, y)

In [ ]:
def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X).reshape(x0.shape)
    y_decision = clf.decision_function(X).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.contourf(x0, x1, y_decision, cmap=plt.cm.brg, alpha=0.1)

plot_predictions(polynomial_svm_clf, [-1.5, 2.5, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.show()

However, there are a couple problems with this technique:
- at a low polynomial degree, this method cannot deal with very complex datasets
- at a high polynomial degree, a huge number of features are created making the model too slow

Fortunately, when using SVMs you can apply an almost miraculous mathematical technique called the **kernel trick**. 
- The kernel trick makes it possible to get the same result as if you had added many polynomial features, even with very high-degree polynomials, without actually having to add them. 
- There is **no combinatorial explosion of the number of features** because you don’t actually add any features. 

We'll explain the mathematics behind the Kernel Trick later. For now, let's just see it in action. This trick is implemented by the `SVC` class. The code below trains an SVM classifier using a **tenth-degree polynomial kernel**. 

The hyperparameter `coef0` controls how much the model is influenced by high-degree polynomials versus low-degree polynomials. 

In [ ]:
from sklearn.svm import SVC

degree = 10
coef0 = 1
C = 0.1
poly_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="poly", degree=degree, coef0=coef0, C=C))
    ])
poly_kernel_svm_clf.fit(X, y)

plot_predictions(poly_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.4, -1, 1.5])
plt.title(f"$d={degree},  coef0={coef0},  C={C}$", fontsize=18);

In [ ]:
from sklearn.svm import SVC

degree = 10
coef0 = 10 # increase the influence of the higher-order terms; overfitting
C = 0.1
poly_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="poly", degree=degree, coef0=coef0, C=C))
    ])
poly_kernel_svm_clf.fit(X, y)

plot_predictions(poly_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.4, -1, 1.5])
plt.title(f"$d={degree},  coef0={coef0},  C={C}$", fontsize=18);

## Similarity Features

Similarity features are another approach that can help separate classes that are not otherwise linearly separable. The idea is to add features computed from a **similarity function**, which simply measures **how much each instance resembles a particular landmark.** 

For example, let’s take the 1D dataset discussed earlier:
- Add two landmarks to it at $x_1 = –2$ and $x_1 = 1$
- Then define the similarity function to be the **Gaussian Radial Basis Function (RBF)** with $γ = 0.3$ ($\gamma$ controls how close you have to be to the landmark to start seeing a significant non-zero similarity).
- Use the RBF function to calculate the distance of the existing feature $x_1$ from the landmarks. **These distances become the new non-linear features,** and range from 0 (very far away from the landmark) to 1 (at the landmark). 

Here is the RBF kernel function:

<img src=images_module05/03_guassianrbf.png width="500" />

The function is bell-shaped (Gaussian) and is plotted below:

In [ ]:
def gaussian_rbf(x, landmark, gamma):
    return np.exp(-gamma * np.linalg.norm(x - landmark, axis=1)**2)

gamma1 = 0.9
gamma2 = 0.1
landmark = 0


x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x_g1 = gaussian_rbf(x1s, landmark, gamma1)
x_g2 = gaussian_rbf(x1s, landmark, gamma2)
plt.plot(x1s, x_g1, label='g1=0.9')
plt.plot(x1s, x_g2, label='g2=0.1')
plt.legend()
plt.show()

# x1s.shape

In [ ]:
# let's plot some figures to visualize how this works...

def gaussian_rbf(x, landmark, gamma):
    return np.exp(-gamma * np.linalg.norm(x - landmark, axis=1)**2)

gamma = 0.3
anchor1 = -2
anchor2 = 1

x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x2s = gaussian_rbf(x1s, anchor1, gamma)
x3s = gaussian_rbf(x1s, anchor2, gamma)

XK = np.c_[gaussian_rbf(X1D, anchor1, gamma), gaussian_rbf(X1D, anchor2, gamma)]
yk = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

plt.figure(figsize=(10.5, 4))

plt.subplot(121)
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.scatter(x=[anchor1, anchor2], y=[0, 0], s=150, alpha=0.5, c="red")
plt.plot(X1D[:, 0][yk==0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][yk==1], np.zeros(5), "g^")
plt.plot(x1s, x2s, "g--")
plt.plot(x1s, x3s, "b:")
plt.gca().get_yaxis().set_ticks([0, 0.25, 0.5, 0.75, 1])
plt.xlabel(r"$x_1$", fontsize=20)
plt.ylabel(r"Similarity", fontsize=14)
plt.annotate(r'$\mathbf{x=-1}$',
             xy=(X1D[3, 0], 0),
             xytext=(-0.5, 0.20),
             ha="center",
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=18,
            )
plt.text(anchor1, 0.9, "$x_2$", ha="center", fontsize=20)
plt.text(anchor2, 0.9, "$x_3$", ha="center", fontsize=20)
plt.axis([-4.5, 4.5, -0.1, 1.1])

plt.subplot(122)
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(XK[:, 0][yk==0], XK[:, 1][yk==0], "bs")
plt.plot(XK[:, 0][yk==1], XK[:, 1][yk==1], "g^")
plt.xlabel(r"$x_2$", fontsize=20)
plt.ylabel(r"$x_3$  ", fontsize=20, rotation=0)
plt.annotate(r'$\phi\left(\mathbf{x=-1}\right)$',
             xy=(XK[3, 0], XK[3, 1]),
             xytext=(0.65, 0.50),
             ha="center",
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=18,
            )
plt.annotate(r'$\phi\left(\mathbf{x=1}\right)$',
             xy=(0.06,1),
             xytext=(0.4, 0.97),
             ha="center",
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=18,
            )

plt.plot([-0.1, 1.1], [0.57, -0.1], "r--", linewidth=3)
plt.axis([-0.1, 1.1, -0.1, 1.1])
    
plt.subplots_adjust(right=1)
plt.show()


Let's look at some examples of $x_1$ to see whats happening here:

$x_1=-1$: 
- This point is 1 away from the first landmark, and 2 away from the second landmark.
- Plug these distances into the Gaussin Radial Basis Function and we get:
    - $x_2 = exp(–0.3 × 1^2 ) ≈ 0.74$
    - $x_3 = exp(–0.3 × 2^2 ) ≈ 0.30$
    
$x_1=1$: 
- This point is 3 away from the first landmark, and 0 away from the second landmark.
- Plug these distances into the Gaussin Radial Basis Function and we get:
    - $x_2 = exp(–0.3 × 3^2 ) ≈ 0.067$
    - $x_3 = exp(–0.3 × 0^2 ) ≈ 1.0$ 

In [ ]:
x1_example = X1D[3, 0] # x1 = -1
for landmark in (-2, 1):
    k = gaussian_rbf(np.array([[x1_example]]), np.array([[landmark]]), gamma)
    print("Phi({}, {}) = {}".format(x1_example, landmark, k))

Take these same steps for every datapoint and voila, we have a **linearly separable dataset.**

The simplest approach for choosing landmarks is to **create one at the location of each and every instance in the dataset.** 

- Doing that creates many dimensions and thus increases the chances that the transformed training set will be linearly separable. 

The downside is that a training set with $m$ instances and $n$ features gets transformed into a training set with **$m$ instances and $m$ features (assuming you drop the original features).**

## Gaussian RBF Kernel

Just like the polynomial features method, the similarity features method can be useful with any Machine Learning algorithm, but it may be computationally expensive to compute all the additional features, especially on large training sets. 

Once again the kernel trick does its SVM magic, making it possible to **obtain a similar result as if you had added many similarity features, without actually increasing the dimensionality.** 

Let’s try the SVC class with the Gaussian RBF kernel:

In [ ]:
rbf_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001)) # Low C = Soft Margin!
    ])
rbf_kernel_svm_clf.fit(X, y)

In [ ]:
from sklearn.svm import SVC

gamma1, gamma2 = 0.1, 5 # increasing gamma narrows the bell shape, meaning landmarks have smaller range of influence 
C1, C2 = 0.001, 1000 # C is inversely proportional to the regularization strength
hyperparams = (gamma1, C1), (gamma1, C2), (gamma2, C1), (gamma2, C2)

svm_clfs = []
for gamma, C in hyperparams:
    rbf_kernel_svm_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=gamma, C=C))
        ])
    rbf_kernel_svm_clf.fit(X, y)
    svm_clfs.append(rbf_kernel_svm_clf)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10.5, 7), sharex=True, sharey=True)

for i, svm_clf in enumerate(svm_clfs):
    plt.sca(axes[i // 2, i % 2])
    plot_predictions(svm_clf, [-1.5, 2.45, -1, 1.5])
    plot_dataset(X, y, [-1.5, 2.45, -1, 1.5])
    gamma, C = hyperparams[i]
    plt.title(r"$\gamma = {}, C = {}$".format(gamma, C), fontsize=16)
    if i in (0, 1):
        plt.xlabel("")
    if i in (1, 3):
        plt.ylabel("")

plt.show()

Note that **increasing gamma makes the bell-shaped curve narrower**. 
- with narrower bellcurves, each instance’s range of influence is smaller
- the decision boundary ends up being more irregular, **wiggling around individual instances.** 

Conversely, a small gamma value makes the bell-shaped curve wider: instances have a larger range of influence, and the decision boundary ends up smoother. 

So gamma acts like a **regularization hyperparameter**: 
- if your model is overfitting, you should reduce it; 
- if it is underfitting, you should increase it (similar to the C hyperparameter). 



## Computational Complexity

### Which classifier implementation should you use?

The `LinearSVC` class is based on the `liblinear` library, which implements an optimized algorithm for linear SVMs:
- **It does not support the kernel trick, but it scales almost linearly with the number of training instances and the number of features.** 
- Its training time complexity is roughly $O( m × n )$. 
- It may take longer to train if you require very high precision (controlled by the tolerance hyperparameter ϵ (called tol in Scikit-Learn) 

The `SVC` class is based on the `libsvm` library:
- **SVC implements an algorithm that supports the kernel trick.** 
- The training time complexity is usually between $O( m^2 × n )$ and $O ( m^3 × n )$
- This means it gets **dreadfully slow when the number of training instances gets large** 
- This algorithm is perfect for **complex small or medium-sized training sets**. It scales well with the number of features, especially with sparse features (i.e., when each instance has few nonzero features). In this case, the algorithm scales roughly with the average number of nonzero features per instance. 

<img src=images_module05/04_svmcomp.png width=600 />

Note that the `LinearSVR` and `SVR` models for regression have similar complexity scaling as their `LinearSVC` and `SVC` counterparts above. 

In general:

- `LinearSVC` is a good place to start, especially if your training set is very large or has many features.
- If your training set is not too large, then you can try the `Gaussian RBF` kernel.

The reasoning for the above will become apparent when we compare the computational complexity. 



## Nonlinear SVM Regression

To tackle nonlinear regression tasks, you can use a **kernelized SVM model,** for which we need to use sklearn's `SVR` class. 
- `SVR` class is the regression equivalent of the `SVC` class
- `LinearSVR` class is the regression equivalent of the `LinearSVC` class

The LinearSVR class **scales linearly with the size of the training set** (just like the LinearSVC class).

The SVR class gets much too slow when the training set grows large (just like the SVC class). 

The plots below show SVM Regression on a random quadratic training set, using a second-degree polynomial kernel. There is little regularization in the left plot (i.e., a large C value), and much more regularization in the right plot (i.e., a small C value). 

**Note**: to be future-proof, we set `gamma="scale"`, as this will be the default value in Scikit-Learn 0.22.

In [ ]:
np.random.seed(42)
m = 100
X = 2 * np.random.rand(m, 1) - 1
y = (0.2 + 0.1 * X + 0.5 * X**2 + np.random.randn(m, 1)/10).ravel()
X.shape

plt.scatter(X,y);

In [ ]:
from sklearn.svm import SVR

svm_poly_reg1 = SVR(kernel="poly", degree=2, C=1000, epsilon=0.2, gamma="scale")
svm_poly_reg2 = SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1, gamma="scale")
svm_poly_reg1.fit(X, y)
svm_poly_reg2.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_poly_reg1, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg1.degree, svm_poly_reg1.C, svm_poly_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
plt.sca(axes[1])
plot_svm_regression(svm_poly_reg2, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg2.degree, svm_poly_reg2.C, svm_poly_reg2.epsilon), fontsize=18)
plt.show()

# Training Non-Linear SVMs

As we discussed, the Kernel trick is one of the most powerful methods at your disposal when training nonlinear SVMs because it gives you the benefits of non-linear features without actually increasing the dimensionality of your feature vectors. 

But why is the Kernel trick available only in some SVM implementations and not others? It has to do with how SVM training works under the hood. Let's take a quick detour to see the mathematics that makes this possible. 


## The Dual Problem

Given a constrained optimization problem, known as the **primal problem**, it is possible to express a different but closely related problem, called its **dual problem**. 

The solution to the **dual problem** typically gives a lower bound to the solution of the primal problem, but under some conditions it can even have the same solutions as the primal problem. 

Luckily, the **SVM problem happens to meet these conditions, so you can choose to solve the primal problem or the dual problem**; both will have the same solution. 

**It turns out that for SVMs, the Dual Problem is faster to solve than the Primal one. It also turns out that the Dual Problem lets you implement the Kernel trick, as we will show.** 

The below equation shows the dual form of the linear SVM objective:

<img src=images_module05/05_svmdual.jpg width=600/>

If you recall, the original objective function (i.e. the Primal Problem) for hard-margin SVMs is:

$$ \text{Hard-margin:} \begin{cases} 
      \text{minimize} & \frac{1}{2}w^T \cdot w \\
      \text{subject to} & y^{(i)}(w^T \cdot x^{(i)}+b)\geq 1 \text{ for every training example } 
   \end{cases}
$$

So how do we go from this, to the dual problem? 


## Lagrange Multipliers

When transforming to the Dual problem, the idea is to transform our **constrained** optimization problem into an **unconstrained** one, by moving the constraints into the objective function itself. The method of **Lagrange Multipliers** lets us do this. 


The famous mathematician Louis Lagrange noticed that when we try to solve an optimization problem in the form:

\begin{aligned}
    & \min_{\mathbf{x}}  f(\mathbf{x})
    \\
    & s.t. \ g(\mathbf{x}) = c 
\end{aligned}

The minimum of $f$ is found when its gradient points in the the same direction as the gradient of $g$, or when:

$$\nabla f(\mathbf{x}) = \alpha \nabla g(\mathbf{x})$$


Therefore, if we want to optimize (i.e. find the min or max of) function $f(\mathbf{x})$ subject to the constraint $g(\mathbf{x}) = c$, we set the gradients of the objective function and equality function to equal one another, and solve for:

$$\nabla f(\mathbf{x}) - \alpha \nabla g(\mathbf{x}) = 0$$

Here, the alpha term is called a **lagrange multipler**. It exists because the gradients will likely not have the same magnitude, hence we use a scalar multiplier to make both terms equal. 


What's the intuition behind this? Think of $g(\mathbf{x})$ as a multidimensional surface. Then think of the contours of $f$ intersecting with $g$. It turns out that the maxima or minima of $f$ on $g$ always correspond to a scenario where a countour of $f$ touches $g$ with a gradient that is parallel, i.e. such that the derivative of $f$ in every direction tangent to $g$ vanishes. 

<img src=images_module05/lagrange_multipliers_illustration.png width=600/>


To simplify the method, we observe that if we define a function :
- $\mathcal{L}(\mathbf{x}, \alpha) = f(\mathbf{x}) - \alpha g(\mathbf{x})$

Then its gradient is:
- $\nabla \mathcal{L}(\mathbf{x}, \alpha) = \nabla f(\mathbf{x}) - \alpha \nabla g(\mathbf{x})$

As a result, solving for $\nabla \mathcal{L}(\mathbf{x}, \alpha) = 0$ allows us to find the minimum.

Let’s look at a simple example: Suppose you want to find the values of $x$ and $y$ that:
- minimize  $f( x , y ) = x^2 + 2y$
- subject to an equality constraint : $g( x, y ) = 3 x + 2 y + 1 = 0$ 

Using the Lagrange multipliers method, we start by defining a new function called the Lagrangian (or Lagrange function ):
- $\mathcal{L} ( x , y , α ) = f ( x , y ) – α g ( x , y )$ 
- $\mathcal{L} ( x , y , α ) = ( x^2 + 2 y) – α (3 x + 2 y + 1)$ 

Caclulate partial derivatives:

- $\frac{\partial}{\partial x}\mathcal{L} ( x , y , \alpha )= 2x - 3\alpha$	
- $\frac{\partial}{\partial y}\mathcal{L} ( x , y , \alpha )= 2 + 2\alpha$	
- $\frac{\partial}{\partial \alpha}\mathcal{L} ( x , y , \alpha )= 3x - 3y$	

Set them to 0:

- $2x - 3\alpha = 2 + 2\alpha = 3x - 3y = 0$	

Solve:
- $x = \frac{3}{2}$
- $y = -\frac{11}{4}$
- $\alpha = 1$


This is a **stationary point** of the Lagrangian: 
- A stationary point is a point where **all partial derivatives are equal to zero.**
- So we calculate the partial derivatives of $\mathcal{L} ( x , y , α )$ with regard to $x$ , $y$ , and $α$, and find the points where these derivatives are all equal to zero
- The solutions to the constrained optimization problem (if they exist) must be among these stationary points

Each constraint (in this case just one) is subtracted from the original objective, multiplied by a new variable called a Lagrange multiplier. 

## Lagrange Multipliers and SVMs

However, this method applies **only to equality constraints (which the SVM primal problem does not have).** 

Fortunately, under some regularity conditions (which are respected by the SVM objectives), this method can be generalized to inequality constraints as well (e.g., $3 x + 2 y + 1 ≥ 0)$. 

The generalized Lagrangian for the hard margin problem is given below, where the $α ( i )$ variables are called the **Karush–Kuhn–Tucker (KKT) multipliers**, and they must be greater or equal to zero. 

<img src=images_module05/06_svmdual2.png width=600/>


Just like with the Lagrange multipliers method, you can compute the partial derivatives and locate the **stationary points**. 

If there is a solution, it will necessarily be among the stationary points that respect the **KKT conditions**: 
- Respect the problem’s constraints: $y^{(i)} (\mathbf{w^T}\mathbf{x}^{(i)}+b) \geq 1 \text{ for  } i=1,2...,m$
- Verify $\alpha^{(i)} \geq 0$ or the $i^{(th)}$ constraint must be an active constraint , meaning it must hold by equality: $y^{(i)} (\mathbf{w^T}\mathbf{x}^{(i)}+b) = 1$. 
    - This condition is called the complementary slackness condition. 
    - It implies that either $\alpha^{(i)}=0$ or the $i^{th}$ instance lies on the boundary (it is a support vector).


Note that the KKT conditions are **necessary conditions for a stationary point to be a solution of the constrained optimization problem.** 

Under some conditions, they are also sufficient conditions. Luckily, the SVM optimization problem happens to meet these conditions, so **any stationary point that meets the KKT conditions is guaranteed to be a solution to the constrained optimization problem.** 

We can compute the partial derivatives of the generalized Lagrangian with regard to w and b with the following equation:

<img src=images_module05/07_svmdual3.png width=600/>

Set these to 0 and we get:

<img src=images_module05/08_svmdual4.png width=600/>

Plug these into our generalized Langrangian funtion, do some algebra, and we get our **dual form SVM problem** (note that the derivation below has a slightly different notation):


<img src=images_module05/09_svmdual5.jpg width=600/>


The goal is now to find the vector $\alpha$ that minimizes this function, with $\alpha \geq0$ for all instances. 

<img src=images_module05/06_svmdual2.png width=600/>


Once you find the optimal $\alpha$, you can compute $\mathbf{w}$ using the equation above.

To compute $b$, you can compute the average over all support vectors to get a more stable and precise value:


<img src=images_module05/10_svmdual6.png width=600/>

The main advantages of the Dual Problem over the Primal problem:
- The objective function now depends ONLY on the lagrange multipliers $\hat{\alpha}$
- The dual problem is faster to solve than the primal when the number of training instances is smaller than the number of features
- It allows us to use the **kernel trick**

## Kernelized SVM
Suppose you want to apply a 2nd-degree polynomial transformation to a two-dimensional training set (such as the moons training set), then train a linear SVM classifier on the transformed training set.

The equation below shows the 2nd-degree polynomial mapping function ϕ that you want to apply:

<img src=images_module05/11_pnkernel.jpg width=600 />

Notice that the transformed vector is three-dimensional instead of two-dimensional. 

Now let’s look at what happens to a couple of two-dimensional vectors, $a$ and $b$, if we apply this 2nd-degree polynomial mapping and then compute the **dot product of the transformed vectors**:

<img src=images_module05/12_pnkernel.jpg width=600/>


**The dot product of the transformed vectors is equal to the square of the dot product of the original vectors:** 
- $ϕ(a)^T · ϕ(b) = (a^T · b)^2.$

Now here is the key insight: **if you apply the transformation $ϕ$ to all training instances, then the dual problem  will contain the dot product:**
- $ϕ(x^{(i)})^T · ϕ(x^{(j)})$


But if $ϕ$ is the 2nd-degree polynomial transformation defined above, **then you can replace this dot product of transformed vectors simply by $(a^T · b)^2$.** 
- You don’t actually need to transform the training instances at all: just replace the dot product by its square: 

<img src=images_module05/05_svmdual.jpg width=600/>

The result will be strictly the same as if you went through the trouble of actually transforming the training set then fitting a linear SVM algorithm, but **this trick makes the whole process much more computationally efficient.** This is the essence of the kernel trick.

To sum up: 

- A kernel $K(a,b)$ is **a special function** that returns the result of a dot product performed in another space 
- Since the feature vector $\mathbf{x}$ only ever appears as a dot product within the objective function of the Dual problem, you don't need to actually transform your features, you can merely replace that dot product with the Kernel $K(a,b)$


## Types of Kernels

The function $K ( a , b ) = ( a ⊺ b )^2$ is a **second-degree polynomial kernel**. 

In Machine Learning, a **kernel** is a function capable of computing the dot product $ϕ(a)^⊺ ϕ ( b )$, based only on the original vectors a and b , without having to compute (or even to know about) the transformation $ϕ$ . Below is a list of the most commonly used kernels:


<img src=images_module05/13_kernels.jpg width=600 />

## Making predictions with the kernel trick!
To incoporate the kernel trick into the dual problem, we write:

<img src=images_module05/14_dualpreds.jpg width=600/>


We also now have to incorporate the kernel trick into our prediction (hypothesis) function! 

<!-- If , we apply the kernel trick to our x values, our equations will include ϕ(x(i)), and w will not have the same dimestions as ϕ(x(i))... and then we are screwed. 
 -->

To do this, we can **plug the formula for `w` we calculated above into the decision function for a new instance of `x`**, and you get an equation with only dot products between input vectors. This makes the kernel trick compatible with our prediction function:

<img src=images_module05/15_dualpreds.jpg width=600/>

Remember that **α(i) ≠ 0 ONLY FOR SUPPORT VECTORS! (points that are along the margin)**. 

Looking at this formula, we better understand why SVMs are called **sparse kernel machines**: because they only need to compute the kernel function on the support vectors and NOT on all the vectors (training instances).

Of course, we also need to do the same for the bias term:

<img src=images_module05/16_dualpreds.jpg width=600/>

## CVXOPT Implementation

- https://xavierbourretsicotte.github.io/SVM_implementation.html

# Online SVMs
Let’s take a quick look at online SVM classifiers:
- recall that online learning means learning incrementally, typically as new instances arrive.

For linear SVM classifiers, one method is to use Gradient Descent (e.g., using SGDClassifier) to minimize the cost function, which is derived from the primal problem. 
- Unfortunately it **converges much more slowly than the methods based on QP**.

<img src=images_module05/17_onlinesvm.jpg width=600/>


**NOTE:** this function is pretty much exactly the same as the Langrangian function we calculated above, except for the 'max' function 
- The first sum in the cost function will push the model to have a small weight vector w, leading to a larger margin. 
- The second sum computes the total of all margin violations. 
- An instance’s margin violation is equal to 0 if it is located off the street and on the correct side, or else it is proportional to the distance to the correct side of the street.  Minimizing this term ensures that the model makes the margin violations as small and as few as possible.

In [ ]:
t = np.linspace(-2, 4, 200)
h = np.where(1 - t < 0, 0, 1 - t)  # max(0, 1-t)

plt.figure(figsize=(5,2.8))
plt.plot(t, h, "b-", linewidth=2, label="$max(0, 1 - t)$")
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.yticks(np.arange(-1, 2.5, 1))
plt.xlabel("$t$", fontsize=16)
plt.axis([-2, 4, -1, 2.5])
plt.legend(loc="upper right", fontsize=16)
plt.show()

Let's implement this from scratch and show it in action by using it to classify the Iris dataset. 

We'll transform this into a binary classification problem with two classes: "Iris virginica" and "Not Iris Virginica". 

In [ ]:
iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = iris["target"]


# show the distribution along these two features
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^", label="Iris virginica")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs", label="Not Iris virginica")
plt.xlabel("Petal length", fontsize=14)
plt.legend(loc="upper left");

In [ ]:
# now make sure dataset has the right shape
X = iris["data"][:, (2, 3)] # petal length, petal width
y = (iris["target"] == 2).astype(np.float64).reshape(-1, 1) # Iris virginica
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.base import BaseEstimator
# This uses gradient descent (with sub gradients)
# http://www.robots.ox.ac.uk/~az/lectures/ml/lect2.pdf

class MyLinearSVC(BaseEstimator):
    def __init__(self, C=1, eta0=1, eta_d=10000, n_epochs=1000, random_state=None):
        self.C = C
        self.eta0 = eta0
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.eta_d = eta_d

    def eta(self, epoch):
        return self.eta0 / (epoch + self.eta_d)
        
    def fit(self, X, y):
        # Random initialization
        if self.random_state:
            np.random.seed(self.random_state)
        w = np.random.randn(X.shape[1], 1) # n feature weights
        b = 0

        m = len(X)
        # Convert classes to -1 and 1
        t = y * 2 - 1  # -1 if t==0, +1 if t==1
        X_t = X * t
        self.Js=[]

        # Training
        for epoch in range(self.n_epochs):
            support_vectors_idx = (X_t.dot(w) + t * b < 1).ravel()
            X_t_sv = X_t[support_vectors_idx]
            t_sv = t[support_vectors_idx]

            J = 1/2 * np.sum(w * w) + self.C * (np.sum(1 - X_t_sv.dot(w)) - b * np.sum(t_sv))
            self.Js.append(J)

            w_gradient_vector = w - self.C * np.sum(X_t_sv, axis=0).reshape(-1, 1)
            b_derivative = -C * np.sum(t_sv)
                
            w = w - self.eta(epoch) * w_gradient_vector
            b = b - self.eta(epoch) * b_derivative
            

        self.intercept_ = np.array([b])
        self.coef_ = np.array([w])
        support_vectors_idx = (X_t.dot(w) + t * b < 1).ravel()
        self.support_vectors_ = X[support_vectors_idx]
        return self

    def decision_function(self, X):
        return X.dot(self.coef_[0]) + self.intercept_[0]

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(np.float64)

    
# train it!
C=100
svm_clf = MyLinearSVC(C=C, eta0 = 10, eta_d = 1000, n_epochs=60000, random_state=2)
svm_clf.fit(X, y)
svm_clf.predict(np.array([[5, 2], [4, 1]]))

In [ ]:
# a quick plot of the loss function versus training epoch
plt.plot(range(svm_clf.n_epochs), svm_clf.Js);
#plt.axis([0, svm_clf.n_epochs, 0, 100])

In [ ]:
# let's view the bias and coefficients that we learned
print(svm_clf.intercept_, svm_clf.coef_)

In [ ]:
# let's compare our 'from-scratch' SGD implementation with scikit learn's
# pre-built SVC classifier
svm_clf2 = SVC(kernel="linear", C=C)
svm_clf2.fit(X, y.ravel())
print(svm_clf2.intercept_, svm_clf2.coef_)

In [ ]:
yr = y.ravel()
fig, axes = plt.subplots(ncols=2, figsize=(11, 3.2), sharey=True)
plt.sca(axes[0])
plt.plot(X[:, 0][yr==1], X[:, 1][yr==1], "g^", label="Iris virginica")
plt.plot(X[:, 0][yr==0], X[:, 1][yr==0], "bs", label="Not Iris virginica")
plot_svc_decision_boundary(svm_clf, 4, 6)
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.title("MyLinearSVC", fontsize=14)
plt.axis([4, 6, 0.8, 2.8])
plt.legend(loc="upper left")

plt.sca(axes[1])
plt.plot(X[:, 0][yr==1], X[:, 1][yr==1], "g^")
plt.plot(X[:, 0][yr==0], X[:, 1][yr==0], "bs")
plot_svc_decision_boundary(svm_clf2, 4, 6)
plt.xlabel("Petal length", fontsize=14)
plt.title("SVC", fontsize=14)
plt.axis([4, 6, 0.8, 2.8])


# Exercises


## Theory
### What is the fundamental idea behind Support Vector Machines?
- The fundamental idea behind Support Vector Machines is to **fit the widest possible “street” between the classes.** 
- The goal is to have the **largest possible margin between the decision boundary that separates the two classes and the training instances.**
- When performing soft margin classification, the SVM searches for a compromise between **perfectly separating the two classes and having the widest possible street (i.e., a few instances may end up on the street).** Another key idea is to use kernels when training on nonlinear datasets.

### What is a support vector?
- AFTER training an SVM, **a support vector is any instance located on the “street”, including its border.** 
- The **decision boundary is entirely determined by the support vectors.** 
- Any instance that is not a support vector (i.e., off the street) has no influence whatsoever; you could remove them, add more instances, or move them around, and as long as they stay off the street they won’t affect the decision boundary. **Computing the predictions only involves the support vectors, not the whole training set.**

### Why is it important to scale the inputs when using SVMs?
- SVMs try to fit the largest possible “street” between the classes, so **if the training set is not scaled, the SVM will tend to neglect small features**.

### Can an SVM classifier output a confidence score when it classifies an instance? What about a probability?
- An SVM classifier can output the distance between the test instance and the decision boundary, and you can use this as a confidence score.
- However, **this score cannot be directly converted into an estimation of the class probability.** 
- If you set probability=True when creating an SVM in Scikit-Learn, then **AFTER training it will calibrate the probabilities using Logistic Regression on the SVM’s scores (trained by an additional five-fold cross-validation on the training data).** 
- This will add the predict_proba() and predict_log_proba() methods to the SVM.

### Should you use the primal or the dual form of the SVM problem to train a model on a training set with millions of instances and hundreds of features?
- **This question applies ONLY TO LINEAR SVMs since kernelized can only use the dual form:**
    - the computational complexity of the primal form of the SVM problem is proportional to the number of training instances m, 
    - the computational complexity of the dual form is proportional to a number between m^2 and m^3. 
- **So if there are millions of instances, you should definitely use the primal form, because the dual form will be much too slow.**

### Say you trained an SVM classifier with an RBF kernel. It seems to underfit the training set: should you increase or decrease γ (gamma)? What about C?
- If an SVM classifier trained with an RBF kernel underfits the training set, there might be too much regularization. To decrease it, you need to:
    - increase gamma
    - increase C
    - increase both.

## Code

_Exercise: Train a `LinearSVC` on a linearly separable dataset. Then train an `SVC` and a `SGDClassifier` on the same dataset. See if you can get them to produce roughly the same model._

Let's use the Iris dataset: the Iris Setosa and Iris Versicolor classes are linearly separable.

In [ ]:
from sklearn import datasets

iris = datasets.load_iris()
X = iris["data"][:, (2, 3)]  # petal length, petal width
y = iris["target"]

setosa_or_versicolor = (y == 0) | (y == 1)
X = X[setosa_or_versicolor]
y = y[setosa_or_versicolor]

In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

C = 5
alpha = 1 / (C * len(X))

lin_clf = LinearSVC(loss="hinge", C=C, random_state=42)
svm_clf = SVC(kernel="linear", C=C)
sgd_clf = SGDClassifier(loss="hinge", learning_rate="constant", eta0=0.001, alpha=alpha,
                        max_iter=1000, tol=1e-3, random_state=42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lin_clf.fit(X_scaled, y)
svm_clf.fit(X_scaled, y)
sgd_clf.fit(X_scaled, y)

print("LinearSVC:                   ", lin_clf.intercept_, lin_clf.coef_)
print("SVC:                         ", svm_clf.intercept_, svm_clf.coef_)
print("SGDClassifier(alpha={:.5f}):".format(sgd_clf.alpha), sgd_clf.intercept_, sgd_clf.coef_)

Let's plot the decision boundaries of these three models:

In [ ]:
# Compute the slope and bias of each decision boundary
w1 = -lin_clf.coef_[0, 0]/lin_clf.coef_[0, 1]
b1 = -lin_clf.intercept_[0]/lin_clf.coef_[0, 1]
w2 = -svm_clf.coef_[0, 0]/svm_clf.coef_[0, 1]
b2 = -svm_clf.intercept_[0]/svm_clf.coef_[0, 1]
w3 = -sgd_clf.coef_[0, 0]/sgd_clf.coef_[0, 1]
b3 = -sgd_clf.intercept_[0]/sgd_clf.coef_[0, 1]

# Transform the decision boundary lines back to the original scale
line1 = scaler.inverse_transform([[-10, -10 * w1 + b1], [10, 10 * w1 + b1]])
line2 = scaler.inverse_transform([[-10, -10 * w2 + b2], [10, 10 * w2 + b2]])
line3 = scaler.inverse_transform([[-10, -10 * w3 + b3], [10, 10 * w3 + b3]])

# Plot all three decision boundaries
plt.figure(figsize=(11, 4))
plt.plot(line1[:, 0], line1[:, 1], "k:", label="LinearSVC")
plt.plot(line2[:, 0], line2[:, 1], "b--", linewidth=2, label="SVC")
plt.plot(line3[:, 0], line3[:, 1], "r-", label="SGDClassifier")
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs") # label="Iris versicolor"
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo") # label="Iris setosa"
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.legend(loc="upper center", fontsize=14)
plt.axis([0, 5.5, 0, 2])

plt.show()

Close enough!

_Exercise: train an SVM classifier on the MNIST dataset. Since SVM classifiers are binary classifiers, you will need to use one-versus-all to classify all 10 digits. You may want to tune the hyperparameters using small validation sets to speed up the process. What accuracy can you reach?_

First, let's load the dataset and split it into a training set and a test set. We could use `train_test_split()` but people usually just take the first 60,000 instances for the training set, and the last 10,000 instances for the test set (this makes it possible to compare your model's performance with others): 

In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, cache=True)

X = mnist["data"]
y = mnist["target"].astype(np.uint8)

X_train = X[:60000]
y_train = y[:60000]
X_test = X[60000:]
y_test = y[60000:]

Many training algorithms are sensitive to the order of the training instances, so it's generally good practice to shuffle them first. However, the dataset is already shuffled, so we do not need to do it.

Let's start simple, with a linear SVM classifier. It will automatically use the One-vs-All (also called One-vs-the-Rest, OvR) strategy, so there's nothing special we need to do. Easy!

**Warning**: this may take a few minutes depending on your hardware.

In [ ]:
lin_clf = LinearSVC(random_state=42)
lin_clf.fit(X_train, y_train)

Let's make predictions on the training set and measure the accuracy (we don't want to measure it on the test set yet, since we have not selected and trained the final model yet):

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = lin_clf.predict(X_train)
accuracy_score(y_train, y_pred)

Okay, 89.5% accuracy on MNIST is pretty bad. This linear model is certainly too simple for MNIST, but perhaps we just needed to scale the data first:

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float32))
X_test_scaled = scaler.transform(X_test.astype(np.float32))

**Warning**: this may take a few minutes depending on your hardware.

In [ ]:
lin_clf = LinearSVC(random_state=42)
lin_clf.fit(X_train_scaled, y_train)

In [ ]:
y_pred = lin_clf.predict(X_train_scaled)
accuracy_score(y_train, y_pred)

That's much better (we cut the error rate by about 25%), but still not great at all for MNIST. If we want to use an SVM, we will have to use a kernel. Let's try an `SVC` with an RBF kernel (the default).

**Note**: to be future-proof we set `gamma="scale"` since it will be the default value in Scikit-Learn 0.22.

In [ ]:
svm_clf = SVC(gamma="scale")
svm_clf.fit(X_train_scaled[:10000], y_train[:10000])

In [ ]:
y_pred = svm_clf.predict(X_train_scaled)
accuracy_score(y_train, y_pred)

That's promising, we get better performance even though we trained the model on 6 times less data. Let's tune the hyperparameters by doing a randomized search with cross validation. We will do this on a small dataset just to speed up the process:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

param_distributions = {"gamma": reciprocal(0.001, 0.1), "C": uniform(1, 10)}
rnd_search_cv = RandomizedSearchCV(svm_clf, param_distributions, n_iter=10, verbose=2, cv=3)
rnd_search_cv.fit(X_train_scaled[:1000], y_train[:1000])

In [ ]:
rnd_search_cv.best_estimator_

In [ ]:
rnd_search_cv.best_score_

This looks pretty low but remember we only trained the model on 1,000 instances. Let's retrain the best estimator on the whole training set (run this at night, it will take hours):

In [ ]:
rnd_search_cv.best_estimator_.fit(X_train_scaled, y_train)

In [ ]:
y_pred = rnd_search_cv.best_estimator_.predict(X_train_scaled)
accuracy_score(y_train, y_pred)

Ah, this looks good! Let's select this model. Now we can test it on the test set:

In [ ]:
y_pred = rnd_search_cv.best_estimator_.predict(X_test_scaled)
accuracy_score(y_test, y_pred)

Not too bad, but apparently the model is overfitting slightly. It's tempting to tweak the hyperparameters a bit more (e.g. decreasing `C` and/or `gamma`), but we would run the risk of overfitting the test set. Other people have found that the hyperparameters `C=5` and `gamma=0.005` yield even better performance (over 98% accuracy). By running the randomized search for longer and on a larger part of the training set, you may be able to find this as well.

_Exercise: train an SVM regressor on the California housing dataset._

Let's load the dataset using Scikit-Learn's `fetch_california_housing()` function:

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = housing["data"]
y = housing["target"]

Split it into a training set and a test set:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Don't forget to scale the data:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Let's train a simple `LinearSVR` first:

In [ ]:
from sklearn.svm import LinearSVR

lin_svr = LinearSVR(random_state=42)
lin_svr.fit(X_train_scaled, y_train)

Let's see how it performs on the training set:

In [ ]:
from sklearn.metrics import mean_squared_error

y_pred = lin_svr.predict(X_train_scaled)
mse = mean_squared_error(y_train, y_pred)
mse

Let's look at the RMSE:

In [ ]:
np.sqrt(mse)

In this training set, the targets are tens of thousands of dollars. The RMSE gives a rough idea of the kind of error you should expect (with a higher weight for large errors): so with this model we can expect errors somewhere around $10,000. Not great. Let's see if we can do better with an RBF Kernel. We will use randomized search with cross validation to find the appropriate hyperparameter values for `C` and `gamma`:

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

param_distributions = {"gamma": reciprocal(0.001, 0.1), "C": uniform(1, 10)}
rnd_search_cv = RandomizedSearchCV(SVR(), param_distributions, n_iter=10, verbose=2, cv=3, random_state=42)
rnd_search_cv.fit(X_train_scaled, y_train)

In [ ]:
rnd_search_cv.best_estimator_

Now let's measure the RMSE on the training set:

In [ ]:
y_pred = rnd_search_cv.best_estimator_.predict(X_train_scaled)
mse = mean_squared_error(y_train, y_pred)
np.sqrt(mse)

Looks much better than the linear model. Let's select this model and evaluate it on the test set:

In [ ]:
y_pred = rnd_search_cv.best_estimator_.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
np.sqrt(mse)